In [ ]:
import os 
os.chdir(r'Q:\sachuriga\Sachuriga_Python/quattrocolo-nwb4fp\src')

from neurochat.nc_data import NData
from neurochat.nc_spike import NSpike
from neurochat.nc_spatial import NSpatial
import neurochat.nc_plot as nc_plot
from neurochat.nc_lfp import NLfp
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBHDF5IO
import matplotlib.pyplot as plt
import numpy as np
import math
import pynapple as nap
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import normalize

import sys
import nwb4fp.analyses.maps as mapp
from nwb4fp.analyses.examples.tracking_plot import plot_ratemap,plot_path
from nwb4fp.analyses.fields import separate_fields_by_laplace, separate_fields_by_dilation,find_peaks,separate_fields_by_laplace_of_gaussian,calculate_field_centers,distance_to_edge_function, remove_fields_by_area, map_pass_to_unit_circle,which_field,compute_crossings
from elephant.statistics import time_histogram, instantaneous_rate
from nwb4fp.analyses import maps
from nwb4fp.analyses.data import pos2speed,speed_filtered_spikes,load_speed_fromNWB,load_units_fromNWB,get_filed_num,unit_location_ch
from scipy.ndimage import gaussian_filter
import ast
import pandas as pd
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=np.inf)
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd


In [ ]:
import pandas as pd
df_loaded = pd.read_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/all_units_table.pkl')
len(df_loaded)
df_good = df_loaded[df_loaded['unit_quality']=="good"]
df_good.head()

In [ ]:
metrics = ['half_width', 'recovery_slope', 'repolarization_slope', 
        'firing_range', 'peak_trough_ratio', 'peak_to_valley','matlab_acg_tau_rise','cell_type_group_2_group']
cls =  ['half_width', 'recovery_slope', 'repolarization_slope', 
        'firing_range', 'peak_trough_ratio', 'peak_to_valley','matlab_acg_tau_rise','cell_type_group_2_group']
df_good = df_good[metrics].dropna()

In [ ]:
import pandas as pd
import numpy as np



# Function to remove outliers using IQR method
def remove_outliers(df, columns, factor=1.5):
    df_clean = df.copy()
    for col in columns:
        if df[col].dtype in [np.float64, np.int64]:  # Only process numeric columns
            Q1 = df[col].quantile(0.25)  # First quartile
            Q3 = df[col].quantile(0.75)  # Third quartile
            IQR = Q3 - Q1  # Interquartile range
            lower_bound = Q1 - factor * IQR
            upper_bound = Q3 + factor * IQR
            # Filter out rows where the value is outside the bounds
            df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
    return df_clean

# Assuming df_good is your original DataFrame
# Remove outliers from numeric columns in metrics
numeric_metrics = [col for col in metrics if df_good[col].dtype in [np.float64, np.int64]]
df_no_outliers = remove_outliers(df_good, numeric_metrics)

# Select only the specified columns and drop any remaining NaN values
df_lda = df_no_outliers[metrics].dropna()

# Optional: Check the shape of the resulting DataFrame
print(f"Original shape: {df_good.shape}")
print(f"Shape after removing outliers and NaN: {df_lda.shape}")

In [ ]:
df_lda 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
dataset=df_lda
# Divide the data set into features (X) and target variable (y)
X = dataset.iloc[:, 0:7].values
y = dataset.iloc[:, 7].values

# Encode the target variable
le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
# Create a pair plot to visualize relationships between different features and species.
ax = sns.pairplot(dataset, hue='cell_type_group_2_group', markers=["o", "s","o", "s"], palette=["blue","red", "cyan", "magenta"],hue_order=[0,2,1,3])
plt.suptitle("Pair Plot of Iris Dataset")
sns.move_legend(
    ax, "lower center",
    bbox_to_anchor=(.5, 1), ncol=3, title=None, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the distribution of each feature using histograms.
plt.figure(figsize=(6, 12))
for i, feature in enumerate(cls[:-1]):
    plt.subplot(4, 2, i + 1)
    sns.histplot(data=dataset, x=feature, hue='cell_type_group_2_group', kde=True)
    plt.title(f'{feature} Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Define custom colors for the 4 groups (you can adjust these hex codes as needed)
colors =["blue","red", "cyan", "magenta"] # Blue, Orange, Green, Red

plt.figure(figsize=(12, 24))
for i, feature in enumerate(cls[:-1]):
    plt.subplot(4, 2, i + 1)
    sns.histplot(
        data=dataset, 
        x=feature, 
        hue='cell_type_group_2_group',
        hue_order=[0, 2, 1, 3],  # Your specified hue order
        palette=colors,         # Applying the custom colors
        kde=True
    )
    plt.title(f'{feature} Distribution')

plt.tight_layout()
plt.show()

In [ ]:
correlation_matrix = dataset.corr(numeric_only = True)
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Split the data set into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Apply Linear Discriminant Analysis
lda = LinearDiscriminantAnalysis(n_components=2)
X_train = lda.fit_transform(X_train, y_train)
X_test = lda.transform(X_test)

In [ ]:
tmp_Df = pd.DataFrame(X_train, columns=['LDA Component 1','LDA Component 2'])
tmp_Df['cell_type_group_2_group']=y_train

sns.FacetGrid(tmp_Df, hue ='cell_type_group_2_group',
              height = 6, hue_order=[0, 2, 1, 3],palette=colors).map(plt.scatter,
                              'LDA Component 1',
                              'LDA Component 2')

plt.legend(loc='upper right')

In [ ]:
classifier = RandomForestClassifier(max_depth=2, random_state=0)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

In [ ]:
#Assume 'y_test' and 'y_pred' are already defined
accuracy = accuracy_score(y_test, y_pred)
conf_m = confusion_matrix(y_test, y_pred)

#Display the accuracy
print(f'Accuracy: {accuracy:.2f}')

#Display the confusion matrix as a heatmap
plt.figure(figsize=(6, 6))
sns.heatmap(conf_m, annot=True, fmt="d", cmap="Blues", cbar=False, square=True)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()